In [0]:



storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(scope="aml-scope", key="storage-access-key")

spark.conf.set("fs.azure.account.key.stbankamldev.blob.core.windows.net", storage_key)

#https://stbankamldev.blob.core.windows.net/raw/rolling_24h_transactions_v2.csv


df_transactions = spark.read.option("header","true").option("inferSchema","true").csv("wasbs://raw@stbankamldev.blob.core.windows.net/transactions.csv")
display(df_transactions)
df_transactions.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date, when

df_transaction_clean = df_transactions\
.withColumn("clean_transaction_date",to_date(col("transaction_date"))) \
.withColumn("clean_transaction_amount",when(col("transaction_amount") == "not_available", None).otherwise(col("transaction_amount").cast("double"))) \
.withColumn("clean_transaction_type",upper(trim(col("transaction_type")))) \
.withColumn("clean_transaction_channel",upper(trim(col("transaction_channel")))) \
.withColumn("clean_transaction_category",upper(trim(col("transaction_category")))) \
.withColumn("clean_transaction_subcategory",upper(trim(col("transaction_subcategory")))) \
.withColumn("clean_counterparty_country",upper(trim(col("counterparty_country")))) \
.withColumn("clean_counterparty_name",upper(trim(col("counterparty_name"))))

df_transaction_clean.filter(col("clean_transaction_amount").isNull()).show()
df_transaction_clean.printSchema()


df_transaction_clean.write.mode("overwrite").parquet("wasbs://silver@stbankamldev.blob.core.windows.net/transaction_clean")

df_transaction_clean_show = spark.read.parquet("wasbs://silver@stbankamldev.blob.core.windows.net/transaction_clean")
df_transaction_clean_show.show()

In [0]:
from pyspark.sql.functions import concat_ws

df_transaction_transformed = df_transaction_clean \
    .withColumn("amount_missing_flag" , when(col("clean_transaction_amount").isNull(), 1 ).otherwise(0)) \
    .withColumn("high_amount_flag", when(col("clean_transaction_amount") >= 10000, 1 ).otherwise(0)) \
    .withColumn("wire_flag", when(col("clean_transaction_type") == "WIRE" ,1).otherwise(0)) \
    .withColumn("high_risk_country_flag", when(col("clean_counterparty_country").isin("IR","RU","AE","CN") , 1).otherwise(0)) \
    .withColumn("transaction_risk_score", (col("amount_missing_flag")+col("high_amount_flag")+col("wire_flag")+col("high_risk_country_flag")).cast("double")) \
    .withColumn("transaction_risk_level", when(col("transaction_risk_score") >=3,"HIGH")
                              .when(col("transaction_risk_score") >=2,"MEDIUM")
                              .when(col("transaction_risk_score") >=1,"LOW")
                              .otherwise("NONE")
                ) 
    
    
df_transaction_transformed.printSchema()
df_transaction_transformed.show()


In [0]:
from pyspark.sql.functions import col, when
df_transaction_alerts = df_transaction_transformed \
.filter(col("transaction_risk_score") >= 3) \
.withColumn("transaction_risk_reason", concat_ws("|",
                                                     when(col("amount_missing_flag") == 1 , "AMOUNT_MISSING"),
                                                     when(col("high_amount_flag") == 1, "HIGH_AMOUNT"),
                                                     when(col("wire_flag") == 1 , "WIRE_TRANSACTION"),
                                                     when(col("high_risk_country_flag") == 1, "HIGH_RISK_COUNTERPARTY_COUNTRY")
                                                     )) \
.select("transaction_id","customer_id","account_id","counterparty_account_id","clean_counterparty_country","clean_transaction_amount","transaction_risk_score",
"transaction_risk_level","transaction_risk_reason")





df_transaction_alerts1 = df_transaction_alerts.dropDuplicates(["transaction_id"])                
display(f"count: {df_transaction_alerts1.count()}")

df_transaction_alerts1.groupBy("transaction_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

df_transaction_alerts1.write.mode("overwrite").parquet("wasbs://gold@stbankamldev.blob.core.windows.net/transaction_alerts")
#https://stbankamldev.blob.core.windows.net/raw/rolling_24h_transactions_v2.csv
display(f"gold transaction alerts count: {df_transaction_alerts1.count()}")

df_transaction_alerts1.show(20, truncate=False)

